# Approach B: Enhanced Classifier

**The 'just add better features' approach:** Same XGBoost, but with temporal patterns, periodicity, trend, and **external data enrichment**. Tests whether the problem is solvable with better feature engineering alone.

**Additional features over Approach A:**
- **Temporal:** weekday/weekend ratio, monthly CV, rolling volatility, trend slope, periodicity strength, consumption entropy
- **Holidays:** holiday consumption mean/std, holiday-to-normal ratio, zero-consumption on holidays (CN holidays via `holidays` package)
- **Weather:** temperature correlation, heating/cooling degree day correlations (simulated seasonal weather)

**Why this matters:** A naive classifier confuses holidays with fraud — both show unusual consumption. By explicitly encoding holiday patterns and weather sensitivity, the model learns what "normal unusual" looks like vs actual theft.

**Expected result:** Improved precision over baseline, fewer false positives from holiday/seasonal effects.

**Infrastructure:** XGBoost classifier trained on Amazon SageMaker AI using SDK v3 (`ModelTrainer`).

In [ ]:
import warnings
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, precision_recall_fscore_support

from utils import (
    get_sagemaker_session, load_sgcc_dataset, preprocess,
    compute_enhanced_features, upload_to_s3,
    train_xgboost_on_sagemaker, download_and_load_model, save_results,
    ROLE, PREFIX,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("Imports OK")

In [ ]:
# SageMaker session
sagemaker_session, region, bucket = get_sagemaker_session()
s3_client = boto3.client("s3")
print(f"Region: {region} | Bucket: {bucket}")

In [ ]:
# Load and preprocess SGCC dataset
raw_df = load_sgcc_dataset()
customer_ids, labels, consumption_clean, date_columns, train_idx, test_idx = preprocess(raw_df)

## Feature Engineering & Training

In [ ]:
# Compute enhanced features (includes all baseline features + temporal/periodicity)
print("Computing enhanced features...")
features_b = compute_enhanced_features(consumption_clean)
features_b["FLAG"] = labels.values
print(f"Feature set: {features_b.shape[1] - 1} features")

# Show the additional features beyond baseline
baseline_cols = {"mean", "std", "max", "min", "range", "median", "skew", "kurtosis", "zero_days", "pct_above_2std"}
enhanced_only = [c for c in features_b.columns if c not in baseline_cols and c != "FLAG"]
print(f"\nEnhanced-only features ({len(enhanced_only)}):")
for col in enhanced_only:
    print(f"  - {col}")

features_b.describe()

In [ ]:
# Split and upload to S3
train_b = features_b.iloc[train_idx]
test_b = features_b.iloc[test_idx]

train_s3 = upload_to_s3(train_b, f"{PREFIX}/approach-b/data/train/train.csv", bucket, s3_client)
test_s3 = upload_to_s3(test_b, f"{PREFIX}/approach-b/data/test/test.csv", bucket, s3_client)

In [ ]:
# Train on SageMaker
trainer_b = train_xgboost_on_sagemaker(
    "approach-b", train_s3, test_s3,
    region=region, bucket=bucket, sagemaker_session=sagemaker_session,
)

In [ ]:
# Load model and evaluate
model_b, metrics_b = download_and_load_model(trainer_b, s3_client)
X_test = test_b.drop(columns=["FLAG"])
y_test = test_b["FLAG"]

y_pred = model_b.predict(X_test)
y_prob = model_b.predict_proba(X_test)[:, 1]

print("\n" + "=" * 60)
print("APPROACH B: ENHANCED CLASSIFIER")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Normal", "Theft"]))

In [ ]:
# Save results for comparison notebook
save_results("approach-b", y_test, y_pred, y_prob, feature_names=list(X_test.columns))

## Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
importances = model_b.feature_importances_
feature_names = list(X_test.columns)
sorted_idx = np.argsort(importances)
colors = ["#f39c12" if feature_names[i] in enhanced_only else "#3498db" for i in sorted_idx]
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color=colors)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel("Feature Importance")
ax.set_title("Approach B: Feature Importance\n(orange = enhanced features, blue = baseline features)")
plt.tight_layout()
plt.show()

## Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=["Normal", "Theft"], yticklabels=["Normal", "Theft"], ax=ax)
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
ax.set_title("Approach B: Confusion Matrix")
plt.tight_layout()
plt.show()

print(f"False Positives: {cm[0,1]} (wasted inspections)")
print(f"True Positives:  {cm[1,1]} (caught theft)")
print(f"False Negatives: {cm[1,0]} (missed theft)")